# 📈 حل مشروع الأسهم والتداول الخوارزمي الذكي (Stock Market Solution)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/stock_market_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
print(df.head())

### 📊 الجزء الأول: التحليل المالي ونسبة شارب

In [ ]:
df['daily_return'] = df.groupby('ticker')['close'].pct_change()

rf = 0.02 / 252 # Daily risk free rate
sharpe = df.groupby('ticker')['daily_return'].apply(
    lambda x: (x.mean() - rf) / x.std() * np.sqrt(252)
)
print("Annualized Sharpe Ratios:")
print(sharpe)

### 🛠️ الجزء الثاني: هندسة الميزات المالية (RSI & Moving Averages)

In [ ]:
aapl = df[df['ticker'] == 'AAPL'].copy()
aapl['SMA_20'] = aapl['close'].rolling(20).mean()
aapl['SMA_50'] = aapl['close'].rolling(50).mean()

# Simple RSI approximation
diff = aapl['close'].diff(1)
gain = diff.where(diff > 0, 0)
loss = -diff.where(diff < 0, 0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / (avg_loss + 1e-9)
aapl['RSI'] = 100 - (100 / (1 + rs))

aapl['Target'] = (aapl['close'].shift(-1) > aapl['close']).astype(int)
aapl = aapl.dropna()

### 🤖 الجزء الثالث: التنبؤ بالاتجاه والمحاكاة (Backtesting)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Features = aapl[['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'SMA_50', 'RSI']]
y = aapl['Target']

# Time-based split for time series evaluation
split_idx = int(len(Features) * 0.8)
X_train, X_test = Features.iloc[:split_idx], Features.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("AAPL Trend Prediction Accuracy:", accuracy_score(y_test, preds))

# Backtesting a simple strategy
test_data = aapl.iloc[split_idx:].copy()
test_data['Pred_Signal'] = preds

# Simple Strategy: Buy if signal = 1, otherwise Hold (or Sell)
test_data['Strategy_Return'] = test_data['daily_return'] * test_data['Pred_Signal'].shift(1)
test_data['Strategy_Return'] = test_data['Strategy_Return'].fillna(0)

cum_hold = (1 + test_data['daily_return']).cumprod() - 1
cum_strat = (1 + test_data['Strategy_Return']).cumprod() - 1

plt.figure(figsize=(10, 6))
plt.plot(test_data['date'], cum_hold, label='Buy & Hold AAPL')
plt.plot(test_data['date'], cum_strat, label='AI Signal Strategy')
plt.title('AAPL Backtesting Performance')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.tight_layout()
plt.savefig('solutions/stock_trading_backtest.png')
plt.show()